In [ ]:
import os
import sys
# add path to custom functions
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/scripts/py_functions")
# import custom functions
from map_plot_tools import *
from line_plot_tools import *
from colorbar_funcs import *
from data_funcs import *
from domain_funcs import nam_domain_outline, core_site_boxes

import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np
#np.set_printoptions(threshold=np.inf) # disable truncation
import pandas as pd 
from scipy.stats import ttest_rel, ttest_ind  # used by sigtest()/sigtest2n() below

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, ListedColormap, LinearSegmentedColormap
from matplotlib import cm
import cmocean.cm as cmo
# settings
%config InlineBackend.figure_format = 'retina'

# top level data directory (override with the WORK_DATA_DIR env var; see config/paths.env.example)
dpath0=os.environ.get('WORK_DATA_DIR', '/glade/work/dervlamk')
# save figs here, under a subdir named for this notebook. HPC has no path to OneDrive, so this
# is a separate directory from the laptop-side FIG_OUTPUT_DIR (config/paths.env.example,
# proxy-side only) -- sync the two by hand (rsync/scp) when picking work back up on the laptop.
# Override the root with HPC_FIG_OUTPUT_DIR; defaults to a figures/ subdir next to the PI/,
# LGM/, obs_data/ dirs under WORK_DATA_DIR.
opath=os.path.join(os.environ.get('HPC_FIG_OUTPUT_DIR', f'{dpath0}/nam-interglacial-dD/notebooks/outputs'))
os.makedirs(opath, exist_ok=True)

In [ ]:
# student's t-test (for data with identical sample sizes)
def sigtest(yearmean1, yearmean2, timemean1, timemean2, alpha=0.05):
    ptvals = ttest_rel(yearmean1, yearmean2, axis=0)
    diff = timemean1 - timemean2
    return diff, mask_insignificant(diff, ptvals.pvalue, alpha), ptvals

# Welch's t-test (for data with different sample sizes)
def sigtest2n(yearmean1, yearmean2, timemean1, timemean2, alpha=0.05):
    ptvals = ttest_ind(yearmean1, yearmean2, axis=0, equal_var=False)
    diff = timemean1 - timemean2
    return diff, mask_insignificant(diff, ptvals.pvalue, alpha), ptvals

def mask_insignificant(diff, pvals, alpha):
    """
    Blank out cells where the difference is not significant at `alpha`, KEEPING the xarray
    structure of `diff` (dims, coords, attrs).

    The previous form -- np.ma.masked_where(pvals > alpha, diff) -- returned a bare numpy
    MaskedArray, which drops every coordinate. That is why
    lgm_pi_diff_mask['OMEGA'][season].sel(lev_p=500.0) raised: after masking there was no
    lev_p coordinate left to select on, so the 3D fields could only be plotted unmasked.

    .where() keeps the DataArray and marks insignificant cells NaN. pcolormesh and quiver
    leave NaN cells blank exactly as they did masked ones, so the figures look the same --
    but .sel(lev_p=...) now works on the masked field too.
    """
    # same dims/coords as diff, carrying the p-values instead of the differences
    pvals = diff.copy(data=pvals)
    pvals.attrs = {'long_name': 'two-tailed p-value', 'units': '1'}
    return diff.where(pvals <= alpha)

def windSpd(u,v):
   windSpd=np.sqrt(u**2 + v**2)
   return windSpd


In [ ]:
# --- LOAD COMPARISON DATA --- #

# IMERG precipitation
climo_filen = f'{dpath0}/obs_data/imerg.gn.2001-2018.climo.nc'
if os.path.exists(climo_filen):
    imerg = xr.open_dataset(climo_filen).precipitation
else:
    filen = f'{dpath0}/obs_data/imerg.gn.timeseries.2001-2018.nc'
    ds = xr.open_dataset(filen).precipitation.transpose('time','lat','lon').groupby("time.month").mean(dim='time') * 24 # convert from mm/hr to mm/day
    ds.attrs['units'] = 'mm/day'
    ds.attrs['Units'] = 'mm/day'
    # Convert lons to 0:360 convention
    imerg = lonFlip(ds)
    imerg.attrs['source_filename'] = 'imerg.gn.timeseries.2001-2018.nc'
    imerg.to_netcdf(climo_filen, mode='w')

# ETOPO05 topography
filen = f'{dpath0}/obs_data/obs.etopo5.zsurf.nc'
etopo_full = xr.open_dataset(f'{filen}').ROSE
etopoSWNA = etopo_full.sel(ETOPO05_X=slice(235,275), ETOPO05_Y=slice(10,42))

# Proxy timeslice mean values
proxydD = pd.read_csv('../data/processed/timeslice_mean_proxy_dDp.csv')

In [ ]:
### +++ SET FILE PATH INFO FOR iCESM1.2 OUTPUT +++ ###

# Points at the per-year (years 801-900) subset produced by
# scripts/nco/subset_tseries.sh + scripts/ncl/pressureRegrid_tseries.ncl, which write into this
# repo's own data/raw/ and data/interim/ (not $WORK_DATA_DIR) -- see data/README.md.
# Both files sit flat in data/raw/ and data/interim/ (no per-case subdirectory),
# named '{varn}.{tag}.0801-0900.tseries.nc' (raw, hybrid-sigma levels) and
# '{varn}.{tag}.0801-0900.tseries.plev.nc' (interim, pressure-level regrid).
# Both the monthly climatology (`dat_climo`, below) and the per-year sample used for significance testing
# (`dat_ts`, below) are derived from this same source, for both cases, so PI and LGM use the same 801-900 window.

raw_varns = ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS',
             'PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS',
             'PRECC', 'PRECL','TS','U','V','OMEGA','Q','Z3','PSL']

files = {}
for case, tag in [('pi', 'iPI.01'), ('lgm', 'i21ka.03')]:
    files[case] = {}
    raw_dir = f'{module_path}/data/raw'
    interim_dir = f'{module_path}/data/interim'
    for varn in raw_varns:   # PS is regrid-only (used by the NCL step), never loaded here
        files[case][varn] = f'{raw_dir}/{varn}.{tag}.0801-0900.tseries.nc'
    for varn in ['U', 'V', 'OMEGA', 'Q', 'Z3']:
        files[case][varn] = f'{interim_dir}/{varn}.{tag}.0801-0900.tseries.plev.nc'   # overrides the hybrid-level path above


In [ ]:
# Canonical identity for every field derive_dat() produces. Stamped explicitly onto each
# output (see the end of the loop below) rather than left to propagate
# Also fills in units/long_name for the five pressure-level fields,
# which arrive from pressureRegrid_tseries.ncl with no attributes at all (ncl vinth2p does not carry them through).
DAT_META = {
    'TS':    {'units': 'K',      'long_name': 'surface temperature (radiative)'},
    'OMEGA': {'units': 'Pa/s',   'long_name': 'vertical pressure velocity'},
    'U':     {'units': 'm/s',    'long_name': 'zonal wind'},
    'V':     {'units': 'm/s',    'long_name': 'meridional wind'},
    'PSL':   {'units': 'Pa',     'long_name': 'sea level pressure'},
    'Q':     {'units': 'kg/kg',  'long_name': 'specific humidity'},
    'Z3':    {'units': 'm',      'long_name': 'geopotential height'},
    'PRECC': {'units': 'mm/day', 'long_name': 'convective precipitation rate'},
    'PRECL': {'units': 'mm/day', 'long_name': 'large-scale precipitation rate'},
    'PRECT': {'units': 'mm/day', 'long_name': 'total precipitation', 'source': 'PRECC + PRECL'},
    'dDp':   {'units': u'‰',     'long_name': u'precipitation-weighted δD of precipitation',
              'source': 'PREC{RC,RL,SC,SL}_HDO over H2O equivalents, weighted by PRECT'},
    'd18Op': {'units': u'‰',     'long_name': u'precipitation-weighted δ18O of precipitation',
              'source': 'PREC{RC,RL,SC,SL}_H218O over H216O equivalents, weighted by PRECT'},
}


def derive_dat(raw, cases, dat_varns, time_dim):
    """Build the dat_varns-keyed derived dict (unit conversions, PRECT, dDp, d18Op) from a
    raw[case][varn] dict. Used once for the climatology (time_dim='month') and once for the
    per-year subset (time_dim='time') -- the isotope/precip-weighting math is identical, only
    the name of the shared time axis differs. Factored out so the dDp/d18Op formula has one
    source, not two copies that can drift apart.

    Every output is stamped with its DAT_META name/units/long_name before being returned, so a
    derived field never carries the identity of whichever raw variable happened to be operand #1.
    """
    dat = {case: {} for case in cases}
    for case in cases:
        for varn in dat_varns:
            if varn in ['TS', 'OMEGA', 'U', 'V', 'PSL', 'Q', 'Z3']:
                dat[case][varn] = raw[case][varn]
            elif varn in ['PRECC', 'PRECL']:
                dat[case][varn] = raw[case][varn]*1000*60*60*24 # convert from m/s to mm/day
            elif varn in ['PRECT']:
                # calculate total precip from convective and large-scale prec vars (snow+rain).
                dat[case][varn] = dat[case]['PRECC'] + dat[case]['PRECL']
            else:
                #== calculate precipitation-weighted fields
                # Weights are each month's share of its OWN year's total precipitation, so they sum to 1 per year on either axis.
                if time_dim == 'month':
                    annual_total_p = dat[case]['PRECT'].sum(dim='month')
                    pWeights = dat[case]['PRECT']/annual_total_p
                else:
                    annual_total_p = dat[case]['PRECT'].groupby('time.year').sum('time')
                    pWeights = dat[case]['PRECT'].groupby('time.year')/annual_total_p
                # the ptiny constant floors the denominator of the isotope ratio equation to prevent divide by zero
                ptiny=1e-18
                if varn=='dDp':
                    # Hydrogen
                    phyd = raw[case]['PRECRC_H2Or'] + raw[case]['PRECRL_H2OR'] + raw[case]['PRECSC_H2Os'] + raw[case]['PRECSL_H2OS']
                    pdeu = raw[case]['PRECRC_HDOr'] + raw[case]['PRECRL_HDOR'] + raw[case]['PRECSC_HDOs'] + raw[case]['PRECSL_HDOS']
                    # replace very small ph values with a tiny value
                    phyd = phyd.where(phyd > ptiny, ptiny)
                    # turn into per mil notation
                    dd = (pdeu/phyd - 1)*1000
                    # Multiply isotope values by weights
                    dat[case][varn] = dd*pWeights
                elif varn=='d18Op':
                    # Oxygen
                    p16o = raw[case]['PRECRC_H216Or'] + raw[case]['PRECRL_H216OR'] + raw[case]['PRECSC_H216Os'] + raw[case]['PRECSL_H216OS']
                    p18o = raw[case]['PRECRC_H218Or'] + raw[case]['PRECRL_H218OR'] + raw[case]['PRECSC_H218Os'] + raw[case]['PRECSL_H218OS']
                    # replace very small ph values with a tiny value
                    p16o = p16o.where(p16o > ptiny, ptiny)
                    # turn into per mil notation
                    do = (p18o/p16o - 1)*1000
                    # Multiply isotope values by weights
                    dat[case][varn] = do*pWeights
                else:
                    # raise, don't print -- an unassigned key surfaces much later as a confusing
                    # KeyError somewhere downstream instead of here.
                    raise KeyError(f'{varn} not recognized by derive_dat()')

            #== stamp identity (see DAT_META above). The .copy(deep=False) is required, not
            # decorative: in the pass-through branch dat[case][varn] IS raw[case][varn], and
            # .rename() returns self when the name is already correct -- so assigning .attrs
            # would clobber the raw array's own CESM metadata (verified: it did, for all seven
            # pass-through vars). deep=False shares the data buffer, so this stays lazy.
            da = dat[case][varn].rename(varn).copy(deep=False)
            da.attrs = dict(DAT_META[varn])
            if time_dim == 'time' and 'year' in da.coords:
                da = da.drop_vars('year')   # left behind by the groupby division above
            dat[case][varn] = da
    return dat

In [ ]:
### +++ PROCESS iCESM1.2 OUTPUT +++ ###

cases=['pi', 'lgm']

#== load per-year raw variables (years 801-900, 1200 monthly records)
raw = {case: {} for case in cases}
for case in cases:
    for varn in raw_varns:
        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
        da = xr.open_dataset(files[case][varn], decode_times=time_coder)[varn]
        # real 'time' dim, noleap calendar (cftime) 
        # CESM h0 timestamps the END of each averaging period (Jan's mean is stamped 0801-02-01),
        # so a raw .dt.month/.dt.year read mislabels every record by one month and
        # misassigns December into the next year.
        da['time'] = da.time - pd.Timedelta(days=2)
        da.attrs['original_time_values'] = da.time
        raw[case][varn] = da


#== construct processed data dictionaries
dat_varns = ['TS', 'OMEGA', 'U', 'V', 'PSL', 'Q', 'Z3', 'PRECC', 'PRECL', 'PRECT', 'dDp', 'd18Op']

# per-year derived data, for significance testing (see the significance-testing section below)
dat_ts = derive_dat(raw, cases, dat_varns, time_dim='time')

# monthly climatology
raw_clim = {case: {varn: raw[case][varn].groupby('time.month').mean('time') for varn in raw_varns} for case in cases}
dat_climo = derive_dat(raw_clim, cases, dat_varns, time_dim='month')
print('Calculated monthly climatologies.')

# seasonal and annual averaging
seasons = ['ann', 'jas'] 
seas_mean     = {case: { varn: {} for varn in dat_varns } for case in cases}
ann_seas_mean = {case: { varn: {} for varn in dat_varns } for case in cases}
for case in cases:
    for varn in dat_varns:
        for season in seasons:
            # get_season() returns 0-based POSITIONAL indices (see its use via .isel(month=...)
            # elsewhere in this notebook and in map_plot_tools.py/line_plot_tools.py) -- not
            # calendar month numbers, so +1 to compare against real dt.month.
            cal_months = [m + 1 for m in get_season(season=season)]
            da = dat_ts[case][varn]
            mask = da['time'].dt.month.isin(cal_months)
            seas_mean[case][varn][season] = da.sel(time=mask).mean(dim='time')
            ann_seas_mean[case][varn][season] = da.sel(time=mask).groupby('time.year').mean(dim='time')
print('Calculated annual and seasonal averages.')


#== iCESM1.2 topography
in_file = '/glade/u/home/dervlamk/nam-interglacial-dD/data/raw/topo_21ka_remap_19x25.mod.170428.sm9.nc'
LANDFR = xr.open_dataset(in_file).LANDFRAC
PHIS = xr.open_dataset(in_file).PHIS
# the available variable is "surface geopotential" in units m2/s2
# approximate the surface geometric elevation by dividing by gravitational acceleration
g = 9.80665 # m/s2
zsurf = PHIS / g
zsurf.attrs['units'] = 'm'
zsurf.attrs['long_name'] = 'surface elevation'
zsurf.attrs['source_file'] = '/glade/work/jiangzhu/data/inputdata/cesm120ka_ICEG6/21ka/topo_21ka_remap_19x25.mod.170428.sm9.nc'

## Statistical significance of LGM$-$PI differences

The cell below runs `sigtest2n()` (Welch's, unpaired -- `pi` and `lgm` are independent
simulations, not paired samples) to get `lgm_pi_diff`/`lgm_pi_diff_mask`/`lgm_pi_ptvals` for
both the 2D surface fields and the 3D pressure-level fields.

In [ ]:
### +++ LGM-PI SIGNIFICANCE TESTING +++ ###

lgm_pi_diff = {varn: {} for varn in dat_varns}
lgm_pi_diff_mask = {varn: {} for varn in dat_varns}
lgm_pi_ptvals = {varn: {} for varn in dat_varns}

print('Significance testing LGM vs PI for:')
for varn in dat_varns:
    print(f'...{varn}')
    for season in seasons:
        diff_, diff_mask_, ptvals_ = sigtest2n(
            ann_seas_mean['lgm'][varn][season], ann_seas_mean['pi'][varn][season],
            seas_mean['lgm'][varn][season], seas_mean['pi'][varn][season]
        )
        lgm_pi_diff[varn][season] = diff_
        lgm_pi_diff_mask[varn][season] = diff_mask_
        lgm_pi_ptvals[varn][season] = ptvals_

print('Done.')

## Core-site box means (LGM$-$PI)

In [ ]:
# --- CORE-SITE MEAN LGM-PI dD_precip --- #
        
# The boxes are defined once, in scripts/py_functions/domain_funcs.py -> core_site_boxes(), and
# are the same object drawn on the maps below (ax.add_geometries(...)) -- so the region shown
# and the region averaged here cannot drift apart. Same box definitions and same cos(lat)
# weighted-mean approach as swna_modern_climatology.ipynb, which averages the OIPC isoscape over
# these boxes. Unlike OIPC (a terrestrial-only isoscape), the model grid has no ocean mask to
# worry about, so this is a plain weighted mean over the box, no NaN-skipping needed.
#
# The model grid is 0:360 in longitude; core_site_boxes() is -180:180 (matching the proxy lon/
# lat columns and how boxes are drawn on these PlateCarree maps), so bounds are converted with
# `% 360` before slicing.

proxy_varns = ['dDp']

site_boxes = core_site_boxes()
site_diff  = {site: { varn:{} for varn in proxy_varns } for site in site_boxes}

for site, poly in site_boxes.items():
    w, s, e, n = poly.bounds  # shapely: (min_lon, min_lat, max_lon, max_lat)
    for varn in proxy_varns:
        for season in seasons:
            box = lgm_pi_diff[varn][season].sel(lon=slice(w % 360, e % 360), lat=slice(s, n))
            weights = np.cos(np.deg2rad(box.lat))
            site_diff[site][varn][season] = float(box.weighted(weights).mean(('lat', 'lon')))

for varn in proxy_varns:
    print(f'Model LGM-PI {varn} [per mil] -- cos(lat)-weighted mean over the averaging box around each core site')
    print(f"{'season':>8}" + ''.join(f'{site:>12}' for site in site_boxes))
    for season in seasons:
        print(f'{season:>8}' + ''.join(f"{site_diff[site][varn][season]:>12.2f}" for site in site_boxes))


# Calculate LGM-Late Holocene dDprecip differences for the proxy records and print summary
core_dDdiff = proxydD['lgm_dD'].values - proxydD['late_holocene_dD'].values

print('\nProxy LGM-LH dDp [per mil] -- at each specific core site')
print(f"{'':>8}" + ''.join(f'{site:>12}' for site in site_boxes))
print(f"{'':>8}" + ''.join(f'{value:>12.2f}' for value in core_dDdiff))

# FIGS

In [ ]:
### +++ USER-DEFINED FIG SETTINGS AND INPUTS +++ ###

# season plotted by the figures below -- select any key in `seasons` (set in the processing cell above)
season = 'jas'

# levels pulled from the 3D pressure-level fields
omega_lev = 500.0   # mb, vertical velocity panel
wind_lev  = 850.0   # mb, wind vectors

# plot specs
bbox     ={'boxstyle':'square','fc':'white','ec':'black','alpha':1,'pad':0.2}
text_kw  ={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1 ={'color':'k', 'weight':'bold', 'size':18, 'ha':'center', 'va':'bottom'}
letters = np.array(['a','b','c'])

# map specs
trans = ccrs.PlateCarree()
proj  = ccrs.PlateCarree()
map_bnds = [-120., -82.5, 10., 36.]

# NAM domain outline, shared with swna_modern_climatology.ipynb (scripts/py_functions/
# domain_funcs.py). site_boxes comes from the core-site box means cell above -- same object,
# so what's drawn here and what's averaged there can't drift apart.
nam_domain = nam_domain_outline()

# vector specs
skip_n=1
w=0.0075
scalef=10
key_length=2

# Lat/Lon vars
core_lons,  core_lats  = proxydD['lon'].values, proxydD['lat'].values
model_lons, model_lats = lgm_pi_diff['PRECT'][season].lon, lgm_pi_diff['PRECT'][season].lat

# construct dictionaries of per-panel model data + color specs
panels = [
    dict(title=r'$\mathbf{\Delta}$ Precipitation',                 # ΔPrecipitation
         data=lgm_pi_diff_mask['PRECT'][season],
         cmap=get_settings(field='precip', diff=True)[0], vmin=-2.5, vmax=2.5, nlevels=21,
         cbar_ticks=[-2, -1, 0, 1, 2], cbar_label='[mm day$^{-1}$]',
         proxy_colored=False, # True to shade core sites on color map
         quiver=False         # True to draw wind vectors
        ),
    dict(title=r'$\mathbf{\Delta}\ \mathbf{\delta D_{precip}}$',   # ΔδD_precip
         data=lgm_pi_diff_mask['dDp'][season],
         cmap=cm.RdBu_r, vmin=-3, vmax=3, nlevels=25,
         cbar_ticks=[-3, -2, -1, 0, 1, 2, 3], cbar_label=u'[‰]',
         proxy_colored=True,  # True to shade core site markers on cmap
         quiver=False         # True to draw wind vectors
        ),
    dict(title=rf'$\mathbf{{\Delta}}\ \mathbf{{\omega_{{{omega_lev:.0f}}}}}$ and {wind_lev:.0f} mb Wind',
         data=lgm_pi_diff_mask['OMEGA'][season].sel(lev_p=omega_lev),
         cmap=cmo.balance, vmin=-0.05, vmax=0.05, nlevels=21,
         cbar_ticks=[-0.04, -0.02, 0, 0.02, 0.04], cbar_label='[Pa s$^{-1}$]',
         proxy_colored=False, # True to shade core site markers on cmap
         quiver=True          # True to draw wind vectors
        ),
]
# colormap normalization
for p in panels:
    p['norm'] = mpl.colors.BoundaryNorm(np.linspace(p['vmin'], p['vmax'], p['nlevels']), p['cmap'].N)

In [ ]:
### +++ LGM-PI CLIMATOLOGY DIFFERENCES FIGURE +++ ###

fig, ax = plt.subplots(nrows=1, ncols=3,
                       figsize=(16,4.5),
                       subplot_kw={'projection': proj},
                       layout='constrained')

# figure title
fig.text(.5,1,' iCESM1.2 LGM (21ka) $-$ PI differences : '+season, **text_kw)

for i, (axi, p) in enumerate(zip(ax, panels)):
    
    # add sub-panel title and label
    axi.text(map_bnds[0]-((map_bnds[0]-map_bnds[1])/2), map_bnds[3]+0.1, p['title'], **text_kw)
    axi.text(map_bnds[0]+1, map_bnds[3]+0.5, letters[i], **text_kw1)
    
    # plot field
    p['cf'] = axi.pcolormesh(model_lons, model_lats, p['data'], cmap=p['cmap'], norm=p['norm'], transform=trans)
    
    # plot vectors
    if p['quiver']:
        # masked, only significant wind changes get an arrow
        u_lev = lgm_pi_diff_mask['U'][season].sel(lev_p=wind_lev)
        v_lev = lgm_pi_diff_mask['V'][season].sel(lev_p=wind_lev)
        q1 = axi.quiver(u_lev.lon[::skip_n], u_lev.lat[::skip_n], u_lev[::skip_n,::skip_n], v_lev[::skip_n,::skip_n],
                         color='k', width=w, scale=scalef, scale_units='inches', units='height',
                         transform=trans, zorder=100)
        axi.quiverkey(q1, .95, 1.035, key_length, rf'{key_length} m/s',
                      labelcolor='k', labelpos='W', fontproperties={'size':9})
        
    # plot core site markers
    if p['proxy_colored']:
        for j, site_data in enumerate(core_dDdiff):
            axi.scatter(x=core_lons[j], y=core_lats[j], c=site_data,
                        cmap=p['cmap'], norm=p['norm'], alpha=1, ec='k', s=150,
                        transform=trans, zorder=100)
            axi.text(core_lons[j]-1.1, core_lats[j], f'{site_data:.1f}‰',
                     fontsize=10, weight='bold', ha='right', bbox=bbox, zorder=100)
        # add averaging area boxes around core sites
        axi.add_geometries(list(site_boxes.values()), crs=trans, fc='none', ec='k', lw=1,
                            linestyle='--', zorder=9)
    else:
        axi.scatter(core_lons, core_lats, ec='k', fc='k', s=150, alpha=1, transform=trans, zorder=100)

    #== map formatting common to all three panels
    # model topography
    axi.contour(zsurf.lon, zsurf.lat, zsurf, levels=np.linspace(500,4000,11), linewidths=0.5, colors='k', transform=trans)
    #axi.contour(zsurf.lon, zsurf.lat, zsurf, levels=[1], linewidths=1.2, colors='k', transform=trans)
    axi.coastlines(lw=1)
    # NAM domain polygon outline
    axi.add_geometries([nam_domain], crs=trans, fc='none', ec='r', lw=2, linestyle='--', zorder=9)
    # grid lines
    axi.set_extent(map_bnds, crs=trans)
    gl = axi.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.top_labels = False; gl.right_labels = False; gl.left_labels = (i == 0)

# colorbars
for i, p in enumerate(panels):
    cax = fig.add_axes([0.05 + i*0.325, 0, 0.275, 0.05])
    cbar = fig.colorbar(p['cf'], ticks=p['cbar_ticks'], orientation='horizontal', extend='both', cax=cax)
    cbar.set_label(p['cbar_label'], weight='normal', labelpad=5, rotation=0)
    cbar.ax.tick_params(labelsize=10)
    for tick in cbar.ax.xaxis.get_major_ticks():
        tick.label1.set_fontweight('normal')

# save output
plt.savefig(os.path.join(opath, "LGM-PI_icesm1p2_diffs.png"), dpi=1200, bbox_inches='tight')

## Convective vs. Large-Scale Precip

In [ ]:
### +++ LGM-PI CONVECTIVE vs. LARGE-SCALE PRECIP DIFFERNECES FIGURE +++ ###

# --- Plot-specific settings --- #
# Map/text/domain specs are deliberately NOT redefined here -- they are reused from the fig
# settings cell so the two figures cannot drift apart.

# both panels share one cmap/norm and one colorbar
pc_cmap, _, _, _ = get_settings(field='precip', diff=True)
pc_vmin = -2.5
pc_vmax = 2.5
pc_levels = 21
pc_norm = mpl.colors.BoundaryNorm(np.linspace(pc_vmin, pc_vmax, pc_levels), pc_cmap.N)

pc_panels = [
    dict(title='PRECC  (convective)',   data=lgm_pi_diff_mask['PRECC'][season]),
    dict(title='PRECL  (large-scale)',  data=lgm_pi_diff_mask['PRECL'][season]),
]

# --- Make plot --- #
fig, ax = plt.subplots(nrows=1, ncols=2,
                       figsize=(11,4.5),
                       subplot_kw={'projection': proj},
                       layout='constrained')

# figure title
fig.text(.5, 1, f' iCESM1.2 LGM (21ka) $-$ PI precipitation partition : {season}', **text_kw)

for i, (axi, p) in enumerate(zip(ax, pc_panels)):

    # add sub-panel title and label
    axi.text(map_bnds[0]-((map_bnds[0]-map_bnds[1])/2), map_bnds[3]+0.1, p['title'], **text_kw)
    axi.text(map_bnds[0]+1, map_bnds[3]+0.5, letters[i], **text_kw1)

    # plot field
    p['cf'] = axi.pcolormesh(model_lons, model_lats, p['data'],
                             cmap=pc_cmap, norm=pc_norm, transform=trans)

    # core site markers
    axi.scatter(core_lons, core_lats, ec='k', fc='k', s=150, alpha=1, transform=trans, zorder=100)

    #== map formatting common to both panels
    # model topography
    axi.contour(zsurf.lon, zsurf.lat, zsurf, levels=np.linspace(500,4000,11), linewidths=0.5, colors='k', transform=trans)
    axi.coastlines(lw=1)
    # NAM domain polygon outline
    axi.add_geometries([nam_domain], crs=trans, fc='none', ec='r', lw=2, linestyle='--', zorder=9)
    # grid lines
    axi.set_extent(map_bnds, crs=trans)
    gl = axi.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.top_labels = False; gl.right_labels = False; gl.left_labels = (i == 0)

# one shared colorbar -- both panels use the same cmap/norm
cax = fig.add_axes([0.15, -.05, 0.7, 0.05])
cbar = fig.colorbar(pc_panels[-1]['cf'], ticks=[-2, -1, 0, 1, 2],
                    orientation='horizontal', extend='both', cax=cax)
cbar.set_label(r'$\Delta$ Precipitation [mm day$^{-1}$]', weight='normal', labelpad=5, rotation=0)
cbar.ax.tick_params(labelsize=10)

# save output
plt.savefig(os.path.join(opath, f'LGM-PI_icesm1p2_diffs_precc_precl.png'), dpi=1200, bbox_inches='tight')

## Moisture Transport

In [ ]:
#=== Calculate moisture transport

qv = { 'pi':{}, 'lgm':{} }
qu = { 'pi':{}, 'lgm':{} }
mt = { 'pi':{}, 'lgm':{} }

for key in ['pi','lgm']:
    qu[key] = dat[key]['U']*dat[key]['Q']
    qv[key] = dat[key]['V']*dat[key]['Q']
    mt[key] = windSpd(qu[key], qv[key]) 